# Redes Neurais

Uma rede neural empilha transformações simples. Cada camada recebe o vetor produzido pela anterior, aplica uma transformação afim e passa o resultado por uma função não linear,

$$
\mathbf{a}^{[l]} = \varphi\left(W^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}\right)
$$

em que o índice $[l]$ identifica a camada, $W^{[l]}$ e $\mathbf{b}^{[l]}$ são os parâmetros dela, $\varphi$ é a função de ativação e $\mathbf{a}^{[l]}$ é a saída. A entrada é a camada zero, $\mathbf{a}^{[0]} = \mathbf{x}$, e a saída da última camada é a previsão do modelo. As camadas intermediárias se chamam camadas ocultas, porque seus valores não aparecem nem na entrada nem na saída.

A ativação parece um detalhe de implementação, mas é ela que sustenta a construção inteira. Sem $\varphi$, duas camadas se reduzem a uma,

$$
\mathbf{z} = W^{[2]}\left(W^{[1]} \mathbf{x} + \mathbf{b}^{[1]}\right) + \mathbf{b}^{[2]}
= \left(W^{[2]} W^{[1]}\right) \mathbf{x} + \left(W^{[2]} \mathbf{b}^{[1]} + \mathbf{b}^{[2]}\right)
$$

que é uma única camada linear com pesos $W^{[2]} W^{[1]}$ e bias $W^{[2]} \mathbf{b}^{[1]} + \mathbf{b}^{[2]}$. Empilhar transformações lineares devolve uma transformação linear, e profundidade sem não-linearidade é ilusória.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import ConfusionMatrixDisplay
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

In [ ]:
# Fixa a semente do gerador de números aleatórios para tornar os resultados reprodutíveis
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

O colapso não é só uma manipulação algébrica. As duas camadas abaixo podem ser substituídas por uma única `nn.Linear` com os pesos e o bias calculados dessa forma, e as duas produzem a mesma saída.

In [ ]:
stacked = nn.Sequential(nn.Linear(2, 8), nn.Linear(8, 2))

equivalent = nn.Linear(2, 2)
with torch.no_grad():
    equivalent.weight.copy_(stacked[1].weight @ stacked[0].weight)
    equivalent.bias.copy_(stacked[1].weight @ stacked[0].bias + stacked[1].bias)

x = torch.randn(5, 2)
print(torch.allclose(stacked(x), equivalent(x), atol=1e-6))

As oito unidades intermediárias não acrescentam nada ao conjunto de funções representáveis. É também por isso que a regressão logística do material anterior não separa a porta XOR: sua fronteira é uma reta, e nenhuma reta deixa as duas diagonais em lados opostos. Acrescentar camadas lineares não resolveria, e acrescentar uma ativação entre elas, sim.

O caminho daqui em diante é montar essa rede em PyTorch, peça por peça: os dados, o modelo, a função de perda, o otimizador e o laço que junta tudo.

## Dataset e DataLoader

O PyTorch separa o carregamento de dados em duas peças. Um `Dataset` responde a duas perguntas, quantos exemplos existem e qual é o exemplo de índice $i$, através dos métodos `__len__` e `__getitem__`. Um `DataLoader` percorre esse conjunto em lotes, cuidando do embaralhamento e da montagem dos tensores.

A classe abaixo é um `Dataset` completo. Ela recebe dois tensores, um com os atributos de cada amostra e outro com os rótulos, e implementa os dois métodos. O `__len__` devolve quantos exemplos existem, e o `__getitem__` devolve o exemplo de um índice, na forma de um par com os atributos e o rótulo.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [ ]:
features = torch.randn(200, 4)
labels = torch.randint(0, 3, (200,))

tabular_dataset = TabularDataset(features, labels)
print(f"exemplos: {len(tabular_dataset)}")

sample_features, sample_label = tabular_dataset[0]
print(f"formato de um exemplo: {tuple(sample_features.shape)}, rótulo: {sample_label.item()}")

Repare que o `Dataset` entrega um exemplo por vez. Quem os agrupa em lotes é o `DataLoader`, que recebe o conjunto e o tamanho do lote, e devolve tensores com uma dimensão a mais na frente, o eixo do lote.

In [ ]:
tabular_dataloader = DataLoader(tabular_dataset, batch_size=32, shuffle=True)

batch_features, batch_labels = next(iter(tabular_dataloader))
print(f"formato do lote: {tuple(batch_features.shape)}, dos rótulos: {tuple(batch_labels.shape)}")

Nada obriga o `Dataset` a guardar os dados em memória. O `__getitem__` pode abrir um arquivo em disco, decodificar uma imagem ou aplicar transformações no exemplo antes de devolvê-lo, e o resto do código continua igual. É essa interface, e não o formato do armazenamento, que o `DataLoader` enxerga.

O problema aqui é o MNIST: 70 mil imagens de dígitos manuscritos, cada uma com $28 \times 28$ pixels em tons de cinza, para classificar em 10 classes. O `torchvision` já o fornece como um `Dataset`, com a mesma interface da classe acima. A transformação `ToTensor` converte cada imagem em um tensor com valores entre 0 e 1, e `Normalize` os centraliza e reescala usando a média e o desvio padrão do conjunto.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

Os 60 mil exemplos de treino são divididos em treino e validação. O treino ajusta os parâmetros, a validação acompanha o aprendizado ao longo das épocas, e o teste fica reservado para uma única avaliação final.

In [ ]:
split_generator = torch.Generator().manual_seed(42)
train_set, validation_set = random_split(full_train_set, [50_000, 10_000], generator=split_generator)

image, label = train_set[0]
print(f"treino: {len(train_set)}, validação: {len(validation_set)}, teste: {len(test_set)}")
print(f"formato da imagem: {tuple(image.shape)}, rótulo: {label}")

Cada imagem chega como um tensor $1 \times 28 \times 28$, ou seja, um canal em tons de cinza, 28 linhas e 28 colunas. Guarde esse formato, porque ele será a entrada do modelo.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index in zip(axes.ravel(), range(10)):
    image, label = train_set[index]
    ax.imshow(image.squeeze(), cmap="gray")
    ax.set_title(f"rótulo {label}")
    ax.axis("off")
plt.show()

No material anterior cada passo do gradiente descendente usava o conjunto inteiro. Com 50 mil exemplos isso daria uma única atualização de parâmetros por passagem pelos dados, cara e pouco frequente. O `DataLoader` divide o conjunto em mini lotes, e cada lote produz um gradiente e uma atualização, de modo que uma passagem completa pelos dados, chamada de época, rende centenas de passos.

O gradiente de um lote é uma estimativa ruidosa do gradiente do conjunto inteiro, e esse ruído em geral ajuda, empurrando os parâmetros para fora de regiões ruins. O embaralhamento a cada época evita que a ordem dos exemplos vire parte do que o modelo aprende, e na validação e no teste ele não faz falta, porque ali não há atualização de parâmetros.

In [ ]:
batch_size = 64

train_dataloader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=1000, shuffle=False)
test_dataloader = DataLoader(test_set, batch_size=1000, shuffle=False)

images, labels = next(iter(train_dataloader))
print(f"lotes de treino por época: {len(train_dataloader)}")
print(f"formato do lote: {tuple(images.shape)}, dos rótulos: {tuple(labels.shape)}")

## Funções de ativação

A ativação $\varphi$ ficou em aberto na formulação da rede. Três funções cobrem a maior parte dos casos,

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
\qquad
\tanh(z) = \frac{e^{z} - e^{-z}}{e^{z} + e^{-z}}
\qquad
\mathrm{ReLU}(z) = \max(0, z)
$$

em que a sigmoid comprime a reta real no intervalo entre 0 e 1, a tanh no intervalo entre $-1$ e 1, e a ReLU zera a parte negativa e deixa a positiva intacta. Todas são aplicadas elemento a elemento e nenhuma altera o formato do tensor.

In [ ]:
z = torch.linspace(-5, 5, 200)

plt.figure(figsize=(8, 5))
plt.plot(z, torch.sigmoid(z), label="sigmoid")
plt.plot(z, torch.tanh(z), label="tanh")
plt.plot(z, torch.relu(z), label="ReLU")
plt.xlabel("z")
plt.ylabel("ativação")
plt.legend()
plt.grid(True)
plt.show()

A sigmoid e a tanh achatam nas duas pontas, ou seja, entradas grandes em módulo produzem saídas quase constantes. Uma unidade nessa região está saturada e deixa de reagir a mudanças nos pesos. A ReLU não satura do lado positivo e custa uma comparação, o que explica a preferência atual por ela. As consequências disso para o treinamento de redes profundas ficam para o material sobre estabilidade.

No PyTorch cada ativação também é um módulo, e `nn.ReLU()` entra na pilha do mesmo jeito que uma `nn.Linear`. A diferença é que ela não tem parâmetros para aprender.

## Construindo modelos em PyTorch

### Camadas e módulos

O `nn.Sequential` recebe uma lista de módulos e os aplica na ordem em que aparecem. Ele já foi usado no material anterior para um modelo de uma camada só, e é o suficiente enquanto o modelo for uma cadeia.

A rede abaixo tem duas camadas com uma ReLU entre elas e resolve a porta XOR. Em vez de treiná-la, seus pesos são escritos à mão, com valores conhecidos que produzem a resposta certa.

In [ ]:
xor_model = nn.Sequential(
    nn.Linear(in_features=2, out_features=2),
    nn.ReLU(),
    nn.Linear(in_features=2, out_features=1),
)

In [ ]:
with torch.no_grad():
    xor_model[0].weight.copy_(torch.tensor([[1.0, 1.0], [1.0, 1.0]]))
    xor_model[0].bias.copy_(torch.tensor([0.0, -1.0]))
    xor_model[2].weight.copy_(torch.tensor([[1.0, -2.0]]))
    xor_model[2].bias.copy_(torch.tensor([0.0]))

In [ ]:
xor_inputs = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])

with torch.no_grad():
    print(xor_model(xor_inputs).squeeze())

print(f"parâmetros: {sum(p.numel() for p in xor_model.parameters())}")

As saídas são 0, 1, 1 e 0, exatamente a porta XOR, com nove parâmetros.

Há duas conclusões aqui, e é importante não confundi-las. A primeira é que a arquitetura representa a função, ou seja, existe um conjunto de pesos que resolve o problema, e é a ativação entre as duas camadas que torna isso possível. A segunda é que nada disso garante que o treinamento vá encontrar esses pesos, já que aqui eles foram escritos à mão.

As mesmas duas peças, empilhadas mais vezes, dão modelos de outra ordem de grandeza. A rede abaixo recebe uma imagem de 28 por 28 pixels, achatada em um vetor de 784 valores pelo `nn.Flatten`, passa por duas camadas ocultas de 2048 unidades e termina em 10 saídas.

In [ ]:
big_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 2048),
    nn.ReLU(),
    nn.Linear(2048, 2048),
    nn.ReLU(),
    nn.Linear(2048, 10),
)

print(f"parâmetros: {sum(p.numel() for p in big_model.parameters()):,}")

De nove parâmetros para quase seis milhões, sem nenhum tipo novo de camada.

O `nn.Sequential` deixa de bastar quando o cálculo não é uma cadeia, como ao reaproveitar a entrada mais adiante, ramificar em dois caminhos ou decidir algo em tempo de execução. A forma geral é herdar de `nn.Module` e escrever a própria classe do modelo.

### A classe do modelo

Uma classe de modelo tem duas partes com papéis bem separados. O construtor `__init__` cria as camadas e as guarda como atributos, ou seja, define quais peças o modelo tem. O `forward` descreve o cálculo, ou seja, recebe um tensor de entrada, faz esse tensor atravessar as camadas na ordem desejada e devolve a saída.

A primeira linha do construtor é sempre `super().__init__()`, a chamada ao construtor do `nn.Module`. É ela que prepara a estrutura interna onde os submódulos e os parâmetros são registrados, e sem ela as atribuições seguintes não são registradas.

Para ver a estrutura sem a distração do MNIST, considere um problema tabular qualquer, com quatro atributos medidos de cada amostra e três classes possíveis. A rede tem uma camada oculta de 16 unidades.

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super().__init__()
        self.hidden = nn.Linear(in_features, hidden_features)
        self.activation = nn.ReLU()
        self.output = nn.Linear(hidden_features, out_features)

    def forward(self, x):
        x = self.hidden(x)        # [batch, 16]
        x = self.activation(x)    # [batch, 16]
        logits = self.output(x)   # [batch, 3]
        return logits

In [ ]:
tabular_model = TabularMLP(in_features=4, hidden_features=16, out_features=3)
print(tabular_model)

Atribuir um módulo a um atributo, como em `self.hidden = nn.Linear(...)`, registra os parâmetros dele no modelo que o contém. É por isso que `parameters()` devolve tudo o que precisa ser otimizado sem que nada seja listado à mão, e é essa lista que será entregue ao otimizador mais adiante.

In [ ]:
for name, parameter in tabular_model.named_parameters():
    print(f"{name}: {tuple(parameter.shape)}")

Repare que o `forward` nunca é chamado diretamente. Escreve-se `model(x)`, e o `nn.Module` executa o `forward` junto com o que mais precisar ser feito antes e depois dele.

### Formatos e parâmetros

Uma camada `nn.Linear(in_features, out_features)` guarda uma matriz de pesos com `out_features` linhas e `in_features` colunas, mais um vetor de `out_features` biases. Acompanhar os formatos camada a camada é a maneira mais rápida de encontrar erros de montagem. A entrada é um lote de amostras, com formato `[batch, 4]`. A camada oculta produz `[batch, 16]`, a ReLU preserva esse formato, e a camada de saída produz `[batch, 3]`.

O total de parâmetros treináveis é

$$
\underbrace{4 \times 16 + 16}_{\text{camada oculta}} + \underbrace{16 \times 3 + 3}_{\text{camada de saída}} = 131 .
$$

In [ ]:
tabular_batch = torch.randn(8, 4)
print(f"entrada: {tuple(tabular_batch.shape)}, saída: {tuple(tabular_model(tabular_batch).shape)}")

calculated_parameters = (4 * 16 + 16) + (16 * 3 + 3)
registered_parameters = sum(p.numel() for p in tabular_model.parameters())
print(f"cálculo manual: {calculated_parameters}, parâmetros registrados: {registered_parameters}")

### Logits

A camada de saída não passa por ativação. O modelo devolve logits, que são valores reais, positivos ou negativos, sem interpretação de probabilidade. Converter logits em probabilidades é tarefa da função de perda, que combina essa conversão com o logaritmo em um cálculo numericamente mais estável do que fazer as duas coisas em separado.

### Exercício: a MLP do MNIST

Escreva uma classe `MNISTMLP`, herdada de `nn.Module`, que classifique os dígitos. A entrada é um lote de imagens com formato `[batch, 1, 28, 28]`, a camada oculta tem 128 unidades com ReLU, e a saída são 10 logits, um por dígito.

Um detalhe precisa de atenção. A `nn.Linear` espera vetores, e a imagem chega com três dimensões. O módulo `nn.Flatten()` colapsa todas as dimensões depois da primeira em uma só, preservando o eixo do lote.

Ao final, imprima o modelo e o número de parâmetros. Quantos são, e de onde vem cada parcela?

In [ ]:
# class MNISTMLP(nn.Module):
#     def __init__(self):
#         super().__init__()
#         ...
#
#     def forward(self, x):
#         ...
#
# model = MNISTMLP().to(device)
# print(model)
# print(f"parâmetros: {sum(p.numel() for p in model.parameters()):,}")

Depois de escrever a classe, descomente a célula abaixo e execute. Ela confere o formato da saída em um lote de verdade, e deve rodar sem erro antes de seguir adiante.

In [ ]:
# images, labels = next(iter(train_dataloader))
# logits = model(images.to(device))
#
# assert logits.shape == (images.shape[0], 10), f"esperado [batch, 10], veio {tuple(logits.shape)}"
# print(f"formato da saída: {tuple(logits.shape)}")

## Funções de perda

A função de perda mede a distância entre a saída do modelo e o alvo, e é o número que o treinamento minimiza. No PyTorch ela também é um módulo, e o uso tem sempre duas etapas: primeiro cria-se a função com os argumentos que a configuram, depois ela é chamada com a previsão e o alvo, nessa ordem. O resultado é um tensor de um único valor, do qual sai o `backward`.

A escolha depende da tarefa. Em regressão, a `nn.MSELoss` calcula a média dos quadrados dos erros.

In [ ]:
mse = nn.MSELoss()

predictions = torch.tensor([2.5, 0.0, 2.1])
targets = torch.tensor([3.0, -0.5, 2.0])

print(f"MSE: {mse(predictions, targets).item():.4f}")

Em classificação binária, a `nn.BCEWithLogitsLoss` recebe o logit produzido pelo modelo, aplica a sigmoid internamente e calcula a entropia cruzada binária. Os alvos são 0.0 ou 1.0, em ponto flutuante e com o mesmo formato dos logits. Nos dois exemplos abaixo o modelo produz o mesmo logit positivo, mas o primeiro alvo é 1 e o segundo é 0, então o segundo está errado e confiante. A perda do lote é a média dos dois.

In [ ]:
bce = nn.BCEWithLogitsLoss()

binary_logits = torch.tensor([2.0, 2.0])
binary_targets = torch.tensor([1.0, 0.0])

print(f"BCE do lote: {bce(binary_logits, binary_targets).item():.4f}")

Para mais de duas classes, a `nn.CrossEntropyLoss` generaliza a entropia cruzada binária, e é ela que aparece no resto desta seção.

Com $C$ classes a última camada produz $C$ logits, e a função que os converte em probabilidades é a softmax,

$$
\hat{p}_c = \frac{e^{z_c}}{\sum_{j=1}^{C} e^{z_j}}
$$

em que a exponencial torna todo valor positivo e a divisão pela soma faz as $C$ probabilidades somarem 1. Com $C = 2$ a softmax se reduz à sigmoid do material anterior.

In [ ]:
example_logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 0.5, 3.0]])
example_targets = torch.tensor([0, 2])

probabilities = torch.softmax(example_logits, dim=1)
print(probabilities)
print(f"somas por linha: {probabilities.sum(dim=1)}")

A perda é a entropia cruzada. Como a classe verdadeira de cada exemplo é uma só, a soma sobre as classes deixa um único termo por exemplo,

$$
J(\theta) = - \frac{1}{m} \sum_{i=1}^{m} \log \hat{p}^{(i)}_{y^{(i)}}
$$

o logaritmo negativo da probabilidade atribuída à classe correta. O custo é baixo quando o modelo dá probabilidade alta à classe certa e cresce sem limite quando ela se aproxima de zero.

In [ ]:
manual = -torch.log(probabilities[range(2), example_targets]).mean()
print(f"cálculo manual: {manual.item():.4f}")
print(f"nn.CrossEntropyLoss: {nn.CrossEntropyLoss()(example_logits, example_targets).item():.4f}")

Os dois valores coincidem. A `nn.CrossEntropyLoss` recebe os logits e faz o caminho inteiro, softmax e logaritmo, em uma expressão estável. Aplicar uma softmax na saída do modelo antes de passá-la à perda é o erro mais comum aqui, e aplica a operação duas vezes.

Repare também no formato dos alvos. A perda recebe uma matriz de logits com formato `[batch, C]` e um vetor de índices inteiros com formato `[batch]`, contendo o número da classe correta de cada exemplo, e não uma codificação one hot.

### Exercício: a perda do MNIST

Escolha a função de perda adequada para classificar os dígitos e atribua-a a `criterion`.

In [ ]:
# criterion = ...

## Otimizadores

O otimizador é quem atualiza os parâmetros a partir dos gradientes. Todos os otimizadores recebem, no primeiro argumento, os parâmetros que devem ser atualizados, quase sempre na forma `model.parameters()`, e a taxa de aprendizado `lr`, que continua sendo o hiperparâmetro mais sensível. Argumentos adicionais configuram a regra de atualização.

O `torch.optim.SGD` aplica o gradiente descendente, $\theta \leftarrow \theta - \eta \nabla_\theta J$. Com o argumento `momentum` ele acumula a direção dos passos anteriores e atravessa regiões planas mais rápido. O `torch.optim.Adam` adapta um passo diferente para cada parâmetro a partir de médias dos gradientes recentes, costuma funcionar bem com pouca calibragem e é a escolha padrão quando não há tempo para ajustar a taxa de aprendizado.

In [ ]:
example_model = nn.Linear(in_features=4, out_features=1)

sgd = torch.optim.SGD(example_model.parameters(), lr=0.1)
sgd_com_momento = torch.optim.SGD(example_model.parameters(), lr=0.1, momentum=0.9)
adam = torch.optim.Adam(example_model.parameters(), lr=0.001)

print(sgd)

O otimizador tem dois métodos usados a cada iteração. O `zero_grad` descarta os gradientes acumulados, e o `step` aplica a regra de atualização aos parâmetros, usando os gradientes que o `backward` deixou guardados.

No exemplo abaixo a perda é a própria soma da saída, cuja derivada em relação ao bias vale 1. Com `lr` igual a 0.1, um passo do SGD subtrai exatamente 0.1 do bias.

In [ ]:
prediction = example_model(torch.randn(1, 4))
loss = prediction.sum()

print(f"bias antes: {example_model.bias.item():.4f}")

sgd.zero_grad()
loss.backward()
sgd.step()

print(f"bias depois: {example_model.bias.item():.4f}")

### Exercício: o otimizador do MNIST

Escolha um otimizador e uma taxa de aprendizado, e atribua o resultado a `optimizer`. Ele precisa receber `model.parameters()`.

In [ ]:
# optimizer = ...

## Loop de treinamento

Com o modelo, a perda e o otimizador definidos, o laço junta tudo. Ele ganha um nível em relação ao material anterior, porque além de percorrer as épocas percorre os mini lotes de cada época. Cada lote é movido para o mesmo dispositivo do modelo com `.to(device)` e passa pelas cinco operações de sempre.

Ao final de cada época a rede é medida no conjunto de validação, que não atualiza parâmetro nenhum. O `model.train()` e o `model.eval()` alternam o modo dos módulos, e quem efetivamente desativa o cálculo de gradientes durante a avaliação é o `torch.no_grad()`. Como a mesma medição será repetida no conjunto de teste, ela vira uma função.

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    total_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            logits = model(images)
            total_loss += criterion(logits, labels).item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

In [ ]:
epochs = 5

train_losses = []
validation_losses = []
validation_accuracies = []

In [ ]:
for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)                    # forward pass
        loss = criterion(logits, labels)          # perda do lote

        optimizer.zero_grad()                     # descarta os gradientes anteriores
        loss.backward()                           # backward pass
        optimizer.step()                          # atualiza os parâmetros

        running_loss += loss.item() * images.size(0)

    validation_loss, validation_accuracy = evaluate(model, validation_dataloader)
    train_losses.append(running_loss / len(train_set))
    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)

    print(f"época {epoch + 1}: perda de treino {train_losses[-1]:.4f}, "
          f"perda de validação {validation_loss:.4f}, acurácia {validation_accuracy:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label="treino")
ax1.plot(validation_losses, label="validação")
ax1.set_xlabel("época")
ax1.set_ylabel("entropia cruzada")
ax1.legend()
ax1.grid(True)

ax2.plot(validation_accuracies)
ax2.set_xlabel("época")
ax2.set_ylabel("acurácia de validação")
ax2.grid(True)
plt.show()

As duas curvas de perda devem cair juntas enquanto a rede aprende padrões que generalizam. Se a perda de treino continuar caindo enquanto a de validação passa a subir, temos um sinal de sobreajuste, ou seja, a rede melhora nos exemplos usados para ajustar os parâmetros mas não em dados que não participaram desse ajuste. É a validação, e não o treino, que diz quando parar.

## Avaliação

O conjunto de teste é usado uma única vez, depois de fechadas todas as escolhas de arquitetura e treinamento. A previsão é a classe de maior logit, que também seria a de maior probabilidade após a softmax.

In [ ]:
test_loss, test_accuracy = evaluate(model, test_dataloader)
print(f"perda de teste: {test_loss:.4f}")
print(f"acurácia de teste: {test_accuracy:.4f}")

Além da métrica final, vale olhar alguns erros. As imagens foram normalizadas na entrada, então a normalização é desfeita antes de exibi-las.

In [ ]:
wrong_examples = []

with torch.no_grad():
    for images, labels in test_dataloader:
        predictions = model(images.to(device)).argmax(dim=1).cpu()

        for index in (predictions != labels).nonzero(as_tuple=True)[0]:
            if len(wrong_examples) < 10:
                wrong_examples.append((images[index], predictions[index].item(), labels[index].item()))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, (image, prediction, label) in zip(axes.ravel(), wrong_examples):
    ax.imshow(image.squeeze() * 0.3081 + 0.1307, cmap="gray")
    ax.set_title(f"previu {prediction}, era {label}")
    ax.axis("off")
plt.show()

Boa parte dos erros é formada por dígitos que uma pessoa também hesitaria em ler. O que a rede não usa é o fato de a entrada ser uma imagem, já que o `nn.Flatten` desmonta a grade de pixels em um vetor, e um dígito deslocado alguns pixels ocupa posições muito diferentes desse vetor. Recuperar a estrutura espacial é o que motiva as redes convolucionais, mais adiante no curso.

## Exercícios

### Exercício 1

Compare a MLP de uma camada oculta de 128 unidades com uma rede de duas camadas ocultas, de 128 e 64 unidades, mantendo os demais hiperparâmetros. Calcule o número de parâmetros de cada uma e desenhe as curvas de validação no mesmo gráfico. A camada adicional trouxe ganho compatível com seu custo?

In [ ]:
hidden_features = [128, 64]

### Exercício 2

Troque o MNIST pelo `datasets.FashionMNIST`, que tem imagens do mesmo formato e também 10 classes. Treine a mesma rede e construa uma matriz de confusão com `ConfusionMatrixDisplay.from_predictions`. Quais pares de classes concentram os erros, e o que as imagens dessas classes têm em comum?

In [ ]:
# Carregue o Fashion-MNIST, treine o modelo e complete a visualização.